In [ ]:
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator

In [ ]:
#Functions to make life simpler
def pca_func(data: pd.DataFrame, title_suffix: str = "") -> PCA:
    """
    Perform PCA and plot explained variance.
    Returns the fitted PCA object and PCA-transformed scores.
    """
    pca = PCA()
    scores = pca.fit_transform(data)

    # Plot explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(pca.explained_variance_, marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()

    # Print summary
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)

    # Return PCA model and score DataFrame
    df_scores = pd.DataFrame(
        scores,
        columns=[f'PC{i}' for i in range(1, scores.shape[1] + 1)],
        index=data.index
    )
    return pca, df_scores

In [ ]:

def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,
           new_axes: bool = False, title: str = "PCA Biplot") -> None:
    """
    Create a PCA biplot showing scores (blue) and variable loadings (red).
    """
    fig, ax = plt.subplots(figsize=(8, 8))

    # Scatter of scores
    ax.scatter(df_scores.PC1, df_scores.PC2, color='b', alpha=0.7)

    # Axis labeling
    if new_axes:
        expl_var = 100 * pca.explained_variance_ratio_
        ax.set_xlabel(f"PC1 ({expl_var[0]:.1f}% explained var.)", fontsize=10)
        ax.set_ylabel(f"PC2 ({expl_var[1]:.1f}% explained var.)", fontsize=10)
    else:
        ax.set_xlabel("PC1", fontsize=10)
        ax.set_ylabel("PC2", fontsize=10)

    # Second axes for loadings
    ax2 = ax.twinx().twiny()
    font = {'color': 'g', 'weight': 'bold', 'size': 10}

    # Draw loading vectors
    for col in df_loadings.columns:
        tipx = df_loadings.loc['PC1', col]
        tipy = df_loadings.loc['PC2', col]
        ax2.arrow(0, 0, tipx, tipy, color='r', alpha=0.5, length_includes_head=True)
        ax2.text(tipx * 1.07, tipy * 1.07, col, fontdict=font, ha='center', va='center')

    # Align and square axes
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')

    plt.title(title)
    plt.tight_layout()


In [ ]:
# Load data
df = pd.read_csv("data/HVs2018_sample.csv", index_col=0)
print("Original Data Preview:")
print(df.head())

# Drop country column (keep only numeric variables)
df = df.iloc[:, 1:]
print("\nNumeric Data Preview:")
print(df.head())


In [ ]:
# Run PCA
pca, df_scores = pca_func(df, " (HVs2018_sample)")

# Compute loadings
df_loadings = pd.DataFrame(
    pca.components_,
    columns=df.columns,
    index=df_scores.columns
)

In [ ]:
# Plot biplot
biplot(df_scores, df_loadings, pca, new_axes=True, title="HVs2018 Sample - PCA Biplot")
plt.show()

In [ ]:
# PCA with Varimax Rotation
# -----------------------------
k=3
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(df)

loadings_df = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2", "PC3"])
print(loadings_df)

In [ ]:
rotator = Rotator(method="varimax")

loadings_rot = rotator.fit_transform(pca.components_.T)
R = rotator.rotation_

load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["RPC1", "RPC2", "RPC3"])
print(load_rot_df)

In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than this value (cut),  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)
print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))
